# Guest Clusters from Reviewer Tags

This notebook clusters guests using a semantic-dominant hybrid of structured reviewer-tag metadata and sentence-embedding representations of the tag text, restricted to the review subset that already has zero-shot topic assignments from `Main_Full_Pipeline`.

It performs five tasks:

1. Parse and collapse reviewer tags into structured guest metadata.
2. Encode the meaning of reviewer tags into guest-level semantic profiles.
3. Benchmark structured-only, semantic-only, and hybrid feature sets to check whether stronger topic separation is real or just an artefact of the clustering representation.
4. Fit the selected semantic-dominant hybrid cluster solution and generate human-readable cluster names from the average profile.
5. Compare topic composition across those clusters with PERMANOVA, a null benchmark, and topic-wise follow-up tests.


In [25]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from guest_clusters_helpers import (
    ZERO_SHOT_MAJOR_TOPICS,
    attach_cluster_names,
    benchmark_cluster_feature_sets,
    bootstrap_cluster_stability,
    build_cluster_assignments,
    build_feature_matrix,
    build_hybrid_feature_matrix,
    build_review_topic_vectors,
    build_semantic_tag_matrix,
    cluster_summary_markdown,
    find_repo_root,
    load_doc_info,
    load_review_data,
    optional_cluster_regressions,
    output_dir_for_repo,
    parse_reviewer_metadata,
    permanova_euclidean,
    prepare_cluster_features,
    reorder_clusters,
    select_cluster_solution,
    summarize_clusters,
    topic_followup_tests,
    topic_separation_vs_null,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
K_VALUES = range(2, 9)
MIN_CLUSTER_SHARE = 0.05
N_BOOTSTRAPS = 30
N_PERMUTATIONS = 999
N_BENCHMARK_PERMUTATIONS = 499

SEMANTIC_MODEL_NAME = "all-MiniLM-L6-v2"
SEMANTIC_MIN_TAG_DF = 20
STRUCTURED_WEIGHT = 1.0
SEMANTIC_WEIGHT = 1.5
ACTIVE_FEATURE_SET = "hybrid_semantic"


ImportError: cannot import name 'attach_cluster_names' from 'guest_clusters_helpers' (c:\Users\fairc\Documents\emat\mdm2\rp\code\guest_clusters_helpers.py)

In [ ]:
repo_root = find_repo_root()
output_dir = output_dir_for_repo(repo_root)

doc_info = load_doc_info(repo_root)
review_df = load_review_data(repo_root, valid_person_ids=doc_info["Person_id"])
metadata = parse_reviewer_metadata(review_df)
feature_df = prepare_cluster_features(metadata)
topic_vectors = build_review_topic_vectors(doc_info)

print(f"Repo root: {repo_root}")
print(f"Output dir: {output_dir}")
print(f"Topic-tagged document rows: {len(doc_info):,}")
print(f"Unique topic-tagged reviews: {doc_info['Person_id'].nunique():,}")
print(f"Metadata rows after filtering to tagged reviews: {len(metadata):,}")
display(metadata[["Person_id", "Trip_Purpose", "Party_Type", "Stay_Bucket", "Room_Count_Bucket", "Submitted_Mobile", "Room_Type_Raw"]].head())


In [ ]:
structured_encoder, structured_matrix, structured_feature_names = build_feature_matrix(feature_df)
semantic_matrix, semantic_tag_summary = build_semantic_tag_matrix(
    metadata,
    model_name=SEMANTIC_MODEL_NAME,
    min_tag_df=SEMANTIC_MIN_TAG_DF,
    use_idf_weighting=True,
    show_progress_bar=True,
)
hybrid_encoder, hybrid_matrix, hybrid_feature_names = build_hybrid_feature_matrix(
    feature_df,
    semantic_matrix,
    structured_weight=STRUCTURED_WEIGHT,
    semantic_weight=SEMANTIC_WEIGHT,
)

feature_matrices = {
    "structured_only": structured_matrix,
    "semantic_only": semantic_matrix,
    ACTIVE_FEATURE_SET: hybrid_matrix,
}
feature_names_by_set = {
    "structured_only": structured_feature_names,
    "semantic_only": [f"semantic_tag_dim_{idx + 1}" for idx in range(semantic_matrix.shape[1])],
    ACTIVE_FEATURE_SET: hybrid_feature_names,
}

feature_benchmark = benchmark_cluster_feature_sets(
    metadata,
    feature_matrices,
    topic_vectors=topic_vectors,
    k_values=K_VALUES,
    min_cluster_share=MIN_CLUSTER_SHARE,
    random_state=RANDOM_STATE,
    topic_permutations=N_BENCHMARK_PERMUTATIONS,
    null_permutations=N_BENCHMARK_PERMUTATIONS,
)
benchmark_display = feature_benchmark[
    [
        "feature_set",
        "k",
        "silhouette_mean",
        "min_cluster_share",
        "topic_r_squared",
        "topic_p_value",
        "topic_null_observed_minus_null_mean_r_squared",
        "topic_null_r_squared_empirical_p",
    ]
].sort_values("topic_null_observed_minus_null_mean_r_squared", ascending=False)
active_feature_matrix = feature_matrices[ACTIVE_FEATURE_SET]
active_feature_names = feature_names_by_set[ACTIVE_FEATURE_SET]

print(f"Active feature set: {ACTIVE_FEATURE_SET}")
print(f"Structured feature dimensions: {structured_matrix.shape[1]}")
print(f"Semantic feature dimensions: {semantic_matrix.shape[1]}")
print(f"Hybrid feature dimensions: {hybrid_matrix.shape[1]}")
display(benchmark_display)
display(semantic_tag_summary.head(20))


In [ ]:
cluster_metrics = select_cluster_solution(
    active_feature_matrix,
    k_values=K_VALUES,
    min_cluster_share=MIN_CLUSTER_SHARE,
    random_state=RANDOM_STATE,
)
selected = cluster_metrics.loc[cluster_metrics["selected"]].iloc[0]
cluster_labels, cluster_centers = reorder_clusters(selected["labels"], selected["model"].cluster_centers_)
assignments = build_cluster_assignments(metadata, cluster_labels)
cluster_summary = summarize_clusters(assignments)
assignments = attach_cluster_names(assignments, cluster_summary)
stability = bootstrap_cluster_stability(
    active_feature_matrix,
    cluster_labels,
    cluster_centers,
    n_clusters=int(selected["k"]),
    n_bootstraps=N_BOOTSTRAPS,
    random_state=RANDOM_STATE,
)

cluster_metrics_display = cluster_metrics.drop(columns=["labels", "model"]).copy()
cluster_metrics_display.insert(0, "feature_set", ACTIVE_FEATURE_SET)
display(cluster_metrics_display)
display(cluster_summary)
display(stability.describe(include="all"))
display(Markdown(cluster_summary_markdown(cluster_summary)))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=cluster_summary, x="Cluster_Name", y="Review_Count", palette="Blues_d", ax=ax)
ax.set_title("Guest cluster sizes (semantic-dominant hybrid)")
ax.set_ylabel("Tagged reviews")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(output_dir / "cluster_sizes.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
analysis_df = assignments.merge(topic_vectors, on="Person_id", how="inner", validate="one_to_one")

topic_matrix = analysis_df[ZERO_SHOT_MAJOR_TOPICS].to_numpy()
permanova_results = permanova_euclidean(
    topic_matrix,
    analysis_df["Cluster_Label"],
    n_permutations=N_PERMUTATIONS,
    random_state=RANDOM_STATE,
)
topic_separation_check = topic_separation_vs_null(
    topic_matrix,
    analysis_df["Cluster_Label"],
    n_permutations=N_PERMUTATIONS,
    random_state=RANDOM_STATE,
)
topic_tests, pairwise_tests = topic_followup_tests(analysis_df)
regression_results = optional_cluster_regressions(analysis_df)

topic_profile = analysis_df.groupby("Cluster_Label")[ZERO_SHOT_MAJOR_TOPICS].mean()
display(permanova_results)
display(topic_separation_check)
display(topic_tests)
if pairwise_tests.empty:
    print("No pairwise topic tests were required after FDR correction.")
else:
    display(pairwise_tests)
if regression_results is None:
    print("statsmodels not available: skipped optional regression robustness section.")
elif regression_results.empty:
    print("Optional regression section ran, but no stable cluster-term estimates were returned.")
else:
    display(regression_results)


In [ ]:
fig_height = max(4.5, 0.65 * len(topic_profile))
fig, ax = plt.subplots(figsize=(11, fig_height))
sns.heatmap(topic_profile, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
ax.set_title("Mean topic proportions by guest cluster (semantic-dominant hybrid)")
ax.set_xlabel("Topic")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(output_dir / "cluster_topic_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
feature_benchmark.to_csv(output_dir / "cluster_feature_benchmark.csv", index=False)
semantic_tag_summary.to_csv(output_dir / "semantic_tag_summary.csv", index=False)
cluster_metrics_display.to_csv(output_dir / "cluster_selection_metrics.csv", index=False)
assignments.to_csv(output_dir / "cluster_assignments.csv", index=False)
cluster_summary.to_csv(output_dir / "cluster_profile_summary.csv", index=False)
stability.to_csv(output_dir / "cluster_stability_metrics.csv", index=False)
permanova_results.to_csv(output_dir / "topic_permanova_results.csv", index=False)
topic_separation_check.to_csv(output_dir / "topic_separation_vs_random.csv", index=False)
topic_tests.to_csv(output_dir / "topic_followup_tests.csv", index=False)
pairwise_tests.to_csv(output_dir / "pairwise_topic_tests.csv", index=False)
if regression_results is not None:
    regression_results.to_csv(output_dir / "topic_cluster_regressions.csv", index=False)
(output_dir / "cluster_profile_summary.md").write_text(cluster_summary_markdown(cluster_summary), encoding="utf-8")

print("Saved outputs:")
for path in sorted(output_dir.iterdir()):
    print(f"- {path.name}")
